# Paper 20 backbone on YOLO26n: FasterNet blocks (+ SimSPPF)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 20:** the C3k2 at layers 6 and 8 use **FasterNet blocks** (PConv) inside; variant 2 also replaces the SPPF at layer 9 with **SimSPPF** |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (the EMA layer of Paper 20 is not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

Two variants are trained:

| Run | Backbone |
|---|---|
| `bb_fasternet` | FasterNet blocks at layers 6 and 8 (Paper 20 model M3) |
| `bb_fasternet_simsppf` | FasterNet blocks at layers 6 and 8 + SimSPPF at layer 9 (Paper 20 model M6) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so fasternet_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 20 backbone: FasterNet block and SimSPPF

**FasterNet block** (Paper 20, Section 3.3, Figure 5):
1. **PConv**: a 3×3 conv on only 1/4 of the channels; the other 3/4 pass through unchanged (fewer FLOPs and less memory access).
2. **MLP**: 1×1 conv to 2× channels (BN + SiLU), then 1×1 conv back to the original channels.
3. Residual add: output = input + MLP(PConv(input)).

Here the FasterNet block replaces the bottleneck *inside* the C3k2 (outer 1×1 convs, split and concat stay the same).

**SimSPPF** (Paper 20, Section 3.3): the same structure as SPPF (k = 5, 3 poolings, same shortcut) with **ReLU** in both
1×1 convs.

**Where they sit in YOLO26n.**

| Layer | Base YOLO26n | `bb_fasternet` | `bb_fasternet_simsppf` | Output |
|---|---|---|---|---|
| 6 | C3k2 (inner C3k) | C3k2 with **FasterNet block** | C3k2 with **FasterNet block** | P4/16 (40 × 40) |
| 8 | C3k2 (inner C3k) | C3k2 with **FasterNet block** | C3k2 with **FasterNet block** | P5/32 (20 × 20) |
| 9 | SPPF (SiLU) | SPPF | **SimSPPF** (ReLU) | P5/32 (20 × 20) |

No layer index changes, so the neck connections stay the same.

**Pretrained weights.** `yolo26n.pt` is loaded first, then the layers are changed:
- layers 6 and 8: the outer `cv1`/`cv2` of the C3k2 keep their pretrained weights; only the FasterNet blocks are new.
  The last 1×1 conv of each FasterNet block starts at zero, so each block starts as an identity mapping (stable start);
- layer 9 (SimSPPF): every weight is kept; only the activation changes.

Paper 20 trained its modified models from scratch while its baselines were pretrained; here every model starts from
`yolo26n.pt`, so the comparison with the baseline is fair.

**Assumptions.** Paper 20 does not give the PConv ratio, MLP ratio or whether the block replaces the whole C3k2 or its
inner bottleneck. Here: PConv on 1/4 of the channels, MLP ratio 2, FasterNet block inside the C3k2.

In [ ]:
%%writefile fasternet_backbone.py
"""Paper 20 backbone for YOLO26n: FasterNet blocks inside the C3k2 at layers 6 and 8; optional SimSPPF (ReLU) at layer 9."""
from copy import deepcopy

import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import SPPF, C3k2
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

FASTER_LAYERS = (6, 8)
SIMSPPF_LAYER = 9


class PConv(nn.Module):
    """Partial convolution: 3x3 conv on the first dim // n_div channels, the rest pass through unchanged."""

    def __init__(self, dim, n_div=4):
        super().__init__()
        self.dc = dim // n_div
        self.du = dim - self.dc
        self.conv = nn.Conv2d(self.dc, self.dc, 3, 1, 1, bias=False)

    def forward(self, x):
        x1, x2 = torch.split(x, [self.dc, self.du], 1)
        return torch.cat((self.conv(x1), x2), 1)


class FasterBlock(nn.Module):
    """FasterNet block: x + 1x1(1x1+BN+SiLU(PConv(x))). The last 1x1 conv starts at zero, so the block starts as identity."""

    def __init__(self, dim, n_div=4, mlp_ratio=2):
        super().__init__()
        self.pconv = PConv(dim, n_div)
        self.mlp = nn.Sequential(Conv(dim, dim * mlp_ratio, 1), nn.Conv2d(dim * mlp_ratio, dim, 1, bias=False))
        nn.init.zeros_(self.mlp[-1].weight)

    def forward(self, x):
        return x + self.mlp(self.pconv(x))


class SimSPPF(SPPF):
    """SPPF with ReLU activations (YOLOv6 SimSPPF). Same weights and forward as SPPF."""


def add_faster(model, layers=FASTER_LAYERS):
    """Replace the inner block(s) of the C3k2 at the given layers with FasterNet blocks; cv1/cv2 keep their weights."""
    for i in layers:
        m = model.model[i]
        assert isinstance(m, C3k2), f"layer {i} is {type(m).__name__}, not C3k2"
        dev = next(m.parameters()).device
        m.m = nn.ModuleList(FasterBlock(m.c) for _ in range(len(m.m))).to(dev)
        m.type = "C3k2_Faster"
        m.np = sum(p.numel() for p in m.parameters())
    return model


def add_simsppf(model, layer=SIMSPPF_LAYER):
    """Turn the SPPF at the given layer into SimSPPF (all weights kept, activations -> ReLU). Returns the model."""
    old = model.model[layer]
    assert type(old) is SPPF, f"layer {layer} is {type(old).__name__}, not SPPF"
    new = deepcopy(old)
    new.__class__ = SimSPPF
    new.cv1.act = nn.ReLU()
    new.cv2.act = nn.ReLU()
    new.type = "SimSPPF"
    model.model[layer] = new
    return model


class FasterTrainer(DetectionTrainer):
    """Loads the pretrained weights first, then FasterNet blocks at layers 6 and 8 (and SimSPPF at layer 9 when SIMSPPF is True)."""

    SIMSPPF = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_faster(model)
        if self.SIMSPPF:
            add_simsppf(model)
        if RANK in {-1, 0}:
            print(f"Paper 20 backbone: FasterNet blocks at layers {FASTER_LAYERS}" + (f", SimSPPF at layer {SIMSPPF_LAYER}" if self.SIMSPPF else ""))
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class FasterSimSPPFTrainer(FasterTrainer):
    SIMSPPF = True


In [ ]:
import importlib, fasternet_backbone
importlib.reload(fasternet_backbone)
from fasternet_backbone import add_faster, add_simsppf, FasterTrainer, FasterSimSPPFTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs the two Paper 20 backbones
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
faster = add_faster(deepcopy(base)).eval()
faster_sim = add_simsppf(add_faster(deepcopy(base))).eval()
with torch.no_grad():
    for m in (faster, faster_sim):
        _ = m(torch.zeros(1, 3, 640, 640))  # forward check
print("layers 6, 8, 9:", [faster_sim.model[i].type for i in (6, 8, 9)])
print("inner blocks of layers 6, 8:", [type(faster.model[i].m[0]).__name__ for i in (6, 8)], "| hidden width c:", [faster.model[i].c for i in (6, 8)])
for tag, m in (("base", base), ("FasterNet", faster), ("FasterNet + SimSPPF", faster_sim)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:20s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

## 4. Training

Two Paper 20 backbone variants are trained with identical settings.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name, trainer)
VARIANTS = [
    ("FasterNet Backbone (Paper 20)",           "bb_fasternet",         FasterTrainer),
    ("FasterNet + SimSPPF Backbone (Paper 20)", "bb_fasternet_simsppf", FasterSimSPPFTrainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run, trainer in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import fasternet_backbone  # must be imported before loading a FasterNet / SimSPPF checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper20_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_fasternet_ncnn_model/`, `bb_fasternet_simsppf_ncnn_model/` (NCNN, imgsz 640) | Raspberry Pi 5 |
| `bb_fasternet.pt`, `bb_fasternet_simsppf.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `fasternet_backbone.py` | Jetson (needed to load the FasterNet / SimSPPF `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper20_backbone_edge_bench.zip paper20_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper20_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("fasternet_backbone.py", EDGE / "fasternet_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import fasternet_backbone  # needed to load the FasterNet / SimSPPF checkpoints
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper20_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper20_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the two Paper 20 backbone models
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_fasternet": None, "bb_fasternet_simsppf": None}
FPS_JETSON = {"bb_fasternet": None, "bb_fasternet_simsppf": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper20_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper20_backbone_edge_bench/` | NCNN + `.pt` models, 50 bench images, `fasternet_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `bb_fasternet_best.pt`, `bb_fasternet_simsppf_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper20_backbone_all_results.zip"
folders = [WORK / "paper20_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")